# Audite as viagens antes de preparar

Uma equipe quer prever a duração de uma viagem antes de ela partir. Cada linha é uma viagem fictícia. O contrato da entrada prevê ID textual, data ISO, código de estação, distância planejada em km, paradas como contagem, duração real em minutos e faturamento em minutos após a chegada. O segundo CSV relaciona códigos e nomes de estações. Comece com os arquivos sintéticos da atividade. Preserve-os. Revise [duplicatas e qualidade](https://devrgs.github.io/Aprendendo-linguagens/#/python/exploracao/qualidade-dos-dados).

Compare duplicata exata com conflito de ID. Retire apenas a cópia idêntica. Se houver dados divergentes com mesmo ID, os critérios do projeto exigem interromper para esclarecimento. Rejeite distâncias negativas e alvos ausentes ou inválidos. Ausência de distância pode ser tratada depois. Zero paradas é válido. Guarde o ID em todas as decisões e use uma junção à esquerda para encontrar estações sem cadastro.

Execute as células na ordem com **Shift+Enter**. A preparação cria uma pasta nova na área de arquivos do caderno. Baixe o caderno editado e os resultados que quiser guardar.


In [ ]:
from pathlib import Path as _Path
from tempfile import mkdtemp as _mkdtemp
import os as _os
import sys as _sys
from importlib import invalidate_caches as _refresh_imports

# Auxiliares editáveis devem ser relidos sem aproveitar bytecode antigo.
_sys.dont_write_bytecode = True

_raiz = _Path.cwd()
_trabalho = _Path(_mkdtemp(prefix='python-auditar-', dir=_raiz))
import numpy
import pandas
import matplotlib


## E1 · Preveja e execute

Leia os dois CSVs e conte observações antes de limpar.


In [ ]:
_pasta = _trabalho / 'e1'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_refresh_imports()


In [ ]:
import pandas as pd
viagens = pd.read_csv("dados/viagens.csv", dtype={"id":"string"})
estacoes = pd.read_csv("dados/estacoes.csv", dtype="string")
print(viagens.shape)
print(estacoes["estacao"].tolist())
print(viagens["id"].iloc[0])


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
(12, 7)
['A', 'B']
V01
```

O cabeçalho descreve sete colunas. A tabela de estações tem duas chaves e a de viagens tem 12 registros brutos.


## E2 · Altere ou depure

Localize a cópia extra, as distâncias ausentes e os zeros de paradas, sem editar os dados.


In [ ]:
_pasta = _trabalho / 'e2'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_refresh_imports()


In [ ]:
import pandas as pd
brutas = pd.read_csv("dados/viagens.csv")
print(brutas.loc[brutas.duplicated(), "id"].tolist())
print(brutas.loc[brutas["km_planejados"].isna(), "id"].tolist())
print(brutas.loc[brutas["paradas_planejadas"].eq(0), "id"].tolist())


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
['V04']
['V03', 'V08']
['V01', 'V08']
```

Zero é uma observação. Ausente é uma lacuna. V08 mostra que uma linha pode ter os dois em colunas diferentes.


## E3 · Resolva

Produza auditoria.json com contagem antes e depois, motivos, unidades e IDs. Faça uma junção que preserve viagens e identifique a estação ausente.

**Pista.** Audite primeiro as cópias idênticas. Depois procure IDs repetidos nos registros únicos e valide o domínio.

Saída esperada

```text
12 10
{'V04': 1}
{'V11': ['km_planejados negativo']}
['V03', 'V08'] ['V08']
```


In [ ]:
_pasta = _trabalho / 'e3'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_refresh_imports()


In [ ]:
import pandas as pd
brutas = pd.read_csv("dados/viagens.csv")
estacoes = pd.read_csv("dados/estacoes.csv")
# Preserve brutas. Registre os motivos de cada exclusão e faça a junção.


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Compare com a solução comentada na aula e faça o caso de borda antes de concluir.
